# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook establishes the **Search Intelligence Data Contract** for **Lane 2: Refresh / Content Opportunity Scoring**. It defines the unit of analysis, verifies data grain and availability via explicit queries, constructs 5 safe pre-decision features, demonstrates the danger of target leakage through a deliberate experiment, and documents slice limitations.

## 1. Unit of analysis + time window

### Plain-Words Data Contract (5 Core Answers):

1. **Unit of Analysis**: One row = One pseudonymized content item (`content_id`) for a specific client (`client_id`) aggregated over a trailing 90-day observation window.
2. **Table(s) Used**: `data/raw/content_refresh_anonymized.csv` (starter dataset slice; aligned with `dim_content` and aggregated `fact_content_daily_performance` tables).
3. **Time Window**: Features are measured over the trailing 90-day window (`impressions_90d`, `clicks_90d`, `sessions_90d`, `days_since_last_update`).
4. **Predicted / Ranked Target**: Proxy target `is_declining_label = (trend_direction == "down")` (Binary 0/1) to output a ranked priority queue of pages needing editorial refresh review.
5. **Deliberately Excluded Field**: `trend_pct` (and `trend_direction` from features). *Reason*: `is_declining_label` is computed directly from `trend_pct` (`trend_direction == "down"`). Including `trend_pct` in feature inputs causes 100% target leakage.

In [1]:
# 1. Load Data & Display Data Contract Summary
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
contract_summary = {
    '1. Grain': '1 Row = 1 Content Item (content_id) per Client (client_id)',
    '2. Primary Source': 'data/raw/content_refresh_anonymized.csv (30,000 rows)',
    '3. Feature Window': 'Trailing 90 Days prior to evaluation point',
    '4. Target Proxy': 'is_declining_label (trend_direction == "down")',
    '5. Excluded Feature': 'trend_pct (Excluded due to 100% target leakage risk)'
}
for k, v in contract_summary.items():
    print(f'{k:20s}: {v}')


1. Grain            : 1 Row = 1 Content Item (content_id) per Client (client_id)
2. Primary Source   : data/raw/content_refresh_anonymized.csv (30,000 rows)
3. Feature Window   : Trailing 90 Days prior to evaluation point
4. Target Proxy     : is_declining_label (trend_direction == "down")
5. Excluded Feature : trend_pct (Excluded due to 100% target leakage risk)


## 2. Fields: feature / label / context / excluded

Every field in our dataset is sorted into one of four strict contract buckets:

| Bucket | Fields | Role / Constraint |
|---|---|---|
| **Feature** | `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`, `days_since_last_update`, `word_count`, `content_age_days`, `scroll_rate`, `engagement_rate` | Observable metrics knowable *before* prediction moment; safe for model inputs. |
| **Label / Proxy** | `is_declining_label` (`trend_direction == "down"`) | Observed outcome target to predict; NEVER used as a model feature. |
| **Context** | `content_id`, `client_id`, `content_type`, `age_tier`, `position_tier` | Used for joining, grouping, stratified splits (`GroupShuffleSplit`), and subgroup analysis; never for direct memorization. |
| **Excluded** | `trend_pct`, `trend_direction` | **EXCLUDED FROM FEATURES**. `trend_pct` directly defines `trend_direction`, creating 100% target leakage. |


In [2]:
# 2. Field Categorization Verification Code
features_list = ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count']
target_col = 'is_declining_label'
context_cols = ['content_id', 'client_id', 'content_type']
excluded_cols = ['trend_pct', 'trend_direction']

print(f'Contract Fields Verified:')
print(f'  - Features ({len(features_list)}) : {features_list}')
print(f'  - Target               : {target_col}')
print(f'  - Context              : {context_cols}')
print(f'  - Excluded (Leakage)   : {excluded_cols}')


Contract Fields Verified:
  - Features (6) : ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count']
  - Target               : is_declining_label
  - Context              : ['content_id', 'client_id', 'content_type']
  - Excluded (Leakage)   : ['trend_pct', 'trend_direction']


## 3. Verify it with queries (grain, counts, missing values, windows)

Here we run three verification checks proving the facts of our data contract.

In [3]:
# Query Fact 1: Grain Uniqueness Verification
# Prove that content_id is unique per row (no duplicate content_ids)
dup_check = df.groupby('content_id').size()
max_dups = dup_check.max()
print(f'Fact 1 (Grain Check): Max rows per content_id = {max_dups}')
assert max_dups == 1, 'Grain violation: duplicate content_ids found!'
print('  -> GRAIN PROOF VERIFIED: One row really is exactly one unique content item.\n')

# Query Fact 2: Slice Row Count & Date/Age Span
total_rows = len(df)
min_age = df['content_age_days'].min()
max_age = df['content_age_days'].max()
print(f'Fact 2 (Row Count & Span): Total dataset rows = {total_rows:,}')
print(f'  -> Content Age Span: {min_age} days to {max_age} days (all content >= 90 days old).\n')

# Query Fact 3: Availability Check (IS TRUE Filter)
has_impressions = (df['impressions_90d'] > 0)
has_valid_age = (df['content_age_days'] >= 90)
surviving_mask = has_impressions & has_valid_age
surviving_rows = surviving_mask.sum()
print(f'Fact 3 (Availability Check):')
print(f'  - Filter (impressions_90d > 0 AND content_age_days >= 90 IS TRUE)')
print(f'  - Surviving rows: {surviving_rows:,} / {total_rows:,} ({surviving_rows/total_rows*100:.1f}% survival rate)')


Fact 1 (Grain Check): Max rows per content_id = 1
  -> GRAIN PROOF VERIFIED: One row really is exactly one unique content item.

Fact 2 (Row Count & Span): Total dataset rows = 30,000
  -> Content Age Span: 90 days to 564 days (all content >= 90 days old).

Fact 3 (Availability Check):
  - Filter (impressions_90d > 0 AND content_age_days >= 90 IS TRUE)
  - Surviving rows: 30,000 / 30,000 (100.0% survival rate)


### Building 5 Features Max (With Availability Statements)

Below we construct a 5-feature matrix $X$ for our lane. Every feature is accompanied by a explicit availability statement proving it is knowable at decision time.

In [4]:
# Construct 5-Feature Frame
feature_df = pd.DataFrame({
    'log_impressions_90d': np.log1p(df['impressions_90d']),
    'days_since_last_update': df['days_since_last_update'].fillna(0),
    'avg_position': df['avg_position'].fillna(0),
    'ctr': df['ctr'].fillna(0),
    'engagement_rate': df['engagement_rate'].fillna(0)
})

df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
y = df['is_declining_label'].values

print('5-Feature Frame Summary:')
display(feature_df.head(5)) if 'display' in globals() else print(feature_df.head(5).to_string())

print('\nAvailability Statements per Feature:')
print('1. log_impressions_90d    : Knowable at decision moment because trailing 90-day search impressions are logged prior to evaluation.')
print('2. days_since_last_update : Knowable at decision moment because content modification timestamps are recorded in the CMS prior to evaluation.')
print('3. avg_position           : Knowable at decision moment because Search Console logs average ranking position over the observation window.')
print('4. ctr                    : Knowable at decision moment because historical click-through-rate is logged in Search Console prior to evaluation.')
print('5. engagement_rate        : Knowable at decision moment because GA4 user engagement rates are logged prior to evaluation.')


5-Feature Frame Summary:
   log_impressions_90d  days_since_last_update  avg_position   ctr  engagement_rate
0             8.243808                      20          10.6  0.76             5.88
1             9.636980                      25          20.3  0.05             0.00
2             9.440023                      20          36.5  0.09             0.00
3             9.371779                      22           6.2  0.49             1.28
4             9.859588                      14          44.0  0.13             0.00

Availability Statements per Feature:
1. log_impressions_90d    : Knowable at decision moment because trailing 90-day search impressions are logged prior to evaluation.
2. days_since_last_update : Knowable at decision moment because content modification timestamps are recorded in the CMS prior to evaluation.
3. avg_position           : Knowable at decision moment because Search Console logs average ranking position over the observation window.
4. ctr                 

### The Trap: Deliberate Target Leakage Experiment

To illustrate the critical danger of target leakage, we deliberately add `trend_pct` (the column from which `trend_direction == "down"` is computed) into our feature set and observe the artificial performance jump.

In [5]:
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# 1. Honest 5-Feature Tree
honest_tree = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
honest_tree.fit(feature_df, y)
honest_scores = honest_tree.predict_proba(feature_df)[:, 1]
honest_p50 = precision_at_k(honest_scores, y, 50)

# 2. Leaky Tree (Adding trend_pct on purpose)
leaky_df = feature_df.copy()
leaky_df['trend_pct'] = df['trend_pct'].fillna(0)

leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
leaky_tree.fit(leaky_df, y)
leaky_scores = leaky_tree.predict_proba(leaky_df)[:, 1]
leaky_p50 = precision_at_k(leaky_scores, y, 50)

print('=== ⚠️ THE TRAP: DELIBERATE LEAKAGE EXPERIMENT ===')
print(f'Honest Model Precision@50 : {honest_p50:.3f} (Realistic evaluation using observable signals)')
print(f'Leaky Model Precision@50  : {leaky_p50:.3f} (Artificial jump caused by feeding the answer in disguise!)')
print('\nLesson: trend_pct encodes the target label directly. Including it teaches the model the definition of the target rather than real world patterns.')
print('-> Removing trend_pct and keeping the honest score: Precision@50 =', f'{honest_p50:.3f}')


=== ⚠️ THE TRAP: DELIBERATE LEAKAGE EXPERIMENT ===
Honest Model Precision@50 : 0.660 (Realistic evaluation using observable signals)
Leaky Model Precision@50  : 1.000 (Artificial jump caused by feeding the answer in disguise!)

Lesson: trend_pct encodes the target label directly. Including it teaches the model the definition of the target rather than real world patterns.
-> Removing trend_pct and keeping the honest score: Precision@50 = 0.660


## 4. Data limits

**Named Data Slice Limitations**:

1. **Unbalanced Client History**: Client tracking start dates differ across the warehouse (`gsc_data_start` vs `ga4_data_start`). Early rows for some clients lack GA4 engagement features (`ga4_data_available = FALSE`).
2. **Observational Nature**: The dataset contains passive observational search performance metrics, not experimental A/B test interventions. We can observe correlations with traffic decline, but cannot guarantee that editing a page causes a traffic recovery.
3. **Proxy Label Limit**: The starter proxy label `trend_direction == "down"` measures current window trend rather than a future non-overlapping outcome window. In Week 4+, we refine this to future-window targets.

In [6]:
# 4. Data Limitation Summary Verification
limitations = [
    '1. Unbalanced Client Tracking History (GA4 data available varies per client)',
    '2. Observational Search Data (Correlations observed, not causal proof)',
    '3. Proxy Target Constraint (Starter trend label vs future 30-day window target)'
]
print('Acknowledged Data Limitations:')
for lim in limitations:
    print(' ', lim)


Acknowledged Data Limitations:
  1. Unbalanced Client Tracking History (GA4 data available varies per client)
  2. Observational Search Data (Correlations observed, not causal proof)
  3. Proxy Target Constraint (Starter trend label vs future 30-day window target)


## Self-check

Before submitting, confirm each item honestly:

- [x] Five plain-words contract answers provided (grain, tables, window, target, excluded field)
- [x] Exactly three verification queries executed with outputs visible (`IS TRUE` availability verified)
- [x] 5-feature frame constructed with an availability statement per feature
- [x] Deliberate target leakage experiment demonstrated (`trend_pct`) and removed
- [x] Data slice limitations explicitly named and documented
- [x] Committed to repo under `work/notebooks/w03_data_contract.ipynb`.